# Test scrapping a full web site

https://docs.scrapy.org/en/latest/topics/jobs.html

In [4]:
from scrapy import Request
from scrapy.crawler import CrawlerProcess
from scrapy.spiders import Spider, CrawlSpider, Rule
from scrapy.linkextractors import LinkExtractor

In [ ]:
class MySpider(scrapy.Spider):
    name = "https://fair-checker.france-bioinformatique.fr"
    allowed_domains = ["fair-checker.france-bioinformatique.fr"]
    start_urls = [
        "https://fair-checker.france-bioinformatique.fr"
    ]

    def parse(self, response):
        self.logger.info("A response from %s just arrived!", response.url)
        print(response.url)

In [ ]:
process = CrawlerProcess()
process.crawl(MySpider)
process.start()
process.stop()

In [5]:
class UrlExtractor(Spider):
    name = 'url-extractor'
    start_urls = []

    def __init__(self, root=None, depth=0, *args, **kwargs):
        self.logger.info("[LE] Source: %s Depth: %s Kwargs: %s", root, depth, kwargs)
        self.source = root
        self.options = kwargs
        self.depth = depth
        UrlExtractor.start_urls.append(root)
        UrlExtractor.allowed_domains = [self.options.get('allow_domains')]
        self.clean_options()
        self.le = LinkExtractor(allow=self.options.get('allow'), deny=self.options.get('deny'),
                                allow_domains=self.options.get('allow_domains'),
                                deny_domains=self.options.get('deny_domains'),
                                restrict_xpaths=self.options.get('restrict_xpaths'),
                                canonicalize=False,
                                unique=True, process_value=None, deny_extensions=None,
                                restrict_css=self.options.get('restrict_css'),
                                strip=True)
        super(UrlExtractor, self).__init__(*args, **kwargs)

    def start_requests(self, *args, **kwargs):
        yield Request('%s' % self.source, callback=self.parse_req)

    def parse_req(self, response):
        all_urls = []
        if int(response.meta['depth']) <= int(self.depth):
            all_urls = self.get_all_links(response)
            for url in all_urls:
                yield Request('%s' % url, callback=self.parse_req)
        if len(all_urls) > 0:
            for url in all_urls:
                yield dict(link=url, meta=dict(source=self.source, depth=response.meta['depth']))

    def get_all_links(self, response):
        links = self.le.extract_links(response)
        str_links = []
        for link in links:
            str_links.append(link.url)
        return str_links

    def clean_options(self):
        allowed_options = ['allow', 'deny', 'allow_domains', 'deny_domains', 'restrict_xpaths', 'restrict_css']
        for key in allowed_options:
            if self.options.get(key, None) is None:
                self.options[key] = []
            else:
                self.options[key] = self.options.get(key).split(',')

In [6]:
process = CrawlerProcess()
process.crawl(UrlExtractor, root='https://fair-checker.france-bioinformatique.fr', depth=0)
process.start()
process.stop()

2024-03-15 14:45:31 [scrapy.utils.log] INFO: Scrapy 2.11.1 started (bot: scrapybot)
2024-03-15 14:45:31 [scrapy.utils.log] INFO: Versions: lxml 4.9.3.0, libxml2 2.11.5, cssselect 1.2.0, parsel 1.9.0, w3lib 2.1.2, Twisted 24.3.0, Python 3.9.17 | packaged by conda-forge | (main, Aug 10 2023, 07:05:25) - [Clang 15.0.7 ], pyOpenSSL 23.2.0 (OpenSSL 3.2.1 30 Jan 2024), cryptography 41.0.3, Platform macOS-14.3.1-x86_64-i386-64bit
2024-03-15 14:45:31 [url-extractor] INFO: [LE] Source: https://fair-checker.france-bioinformatique.fr Depth: 0 Kwargs: {}
2024-03-15 14:45:31 [scrapy.addons] INFO: Enabled addons:
[]
2024-03-15 14:45:31 [py.warnings] WARNING: /Users/gaignard-a/miniconda3/envs/fair-checker/lib/python3.9/site-packages/scrapy/utils/request.py:254: ScrapyDeprecationWarning: '2.6' is a deprecated value for the 'REQUEST_FINGERPRINTER_IMPLEMENTATION' setting.

It is also the default value. In other words, it is normal to get this warning if you have not defined a value for the 'REQUEST_FING

2024-03-15 14:45:32 [scrapy.core.scraper] DEBUG: Scraped from <200 https://fair-checker.france-bioinformatique.fr>
{'link': 'https://github.com/IFB-ElixirFr/fair-checker/issues', 'meta': {'source': 'https://fair-checker.france-bioinformatique.fr', 'depth': 0}}
2024-03-15 14:45:32 [scrapy.core.scraper] DEBUG: Scraped from <200 https://fair-checker.france-bioinformatique.fr>
{'link': 'https://github.com/IFB-ElixirFr/fair-checker', 'meta': {'source': 'https://fair-checker.france-bioinformatique.fr', 'depth': 0}}
2024-03-15 14:45:32 [scrapy.core.scraper] DEBUG: Scraped from <200 https://fair-checker.france-bioinformatique.fr>
{'link': 'https://github.com/IFB-ElixirFr/FAIR-checker/releases/tag/v1.2.7', 'meta': {'source': 'https://fair-checker.france-bioinformatique.fr', 'depth': 0}}
2024-03-15 14:45:32 [scrapy.core.scraper] DEBUG: Scraped from <200 https://fair-checker.france-bioinformatique.fr>
{'link': 'https://datacite.org/', 'meta': {'source': 'https://fair-checker.france-bioinformatiqu

<DeferredList at 0x10dd1c820 current result: []>